# 第 22 章：模型选择与超参数

这个 notebook 用一个小型 Gaia 风格颜色-温度数据集演示：

- 超参数如何控制模型复杂度
- 为什么不能直接用测试集挑模型
- 为什么交叉验证比单次验证切分更稳


In [ ]:
from __future__ import annotations

import csv
import math
from pathlib import Path

DATA_PATH = Path("../../data/small/stellar_teff_model_selection_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "source_id": raw["source_id"],
            "bp_rp": float(raw["bp_rp"]),
            "teff_k": float(raw["teff_k"]),
        })

print(f"Loaded {len(rows)} stellar samples from {DATA_PATH.name}")
print("bp_rp range:", min(row["bp_rp"] for row in rows), "to", max(row["bp_rp"] for row in rows))
print("teff range:", min(row["teff_k"] for row in rows), "to", max(row["teff_k"] for row in rows))


In [ ]:
validation_ids = {"S004", "S009", "S014"}
test_ids = {"S005", "S010", "S015"}

train_rows = [row for row in rows if row["source_id"] not in validation_ids | test_ids]
validation_rows = [row for row in rows if row["source_id"] in validation_ids]
train_validation_rows = [row for row in rows if row["source_id"] not in test_ids]
test_rows = [row for row in rows if row["source_id"] in test_ids]

print("train/validation/test sizes:", len(train_rows), len(validation_rows), len(test_rows))
print("validation ids:", sorted(validation_ids))
print("test ids:", sorted(test_ids))


In [ ]:
def solve_linear_system(matrix, vector):
    size = len(vector)
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]

    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row_index: abs(augmented[row_index][col]))
        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError("Singular matrix encountered while fitting polynomial")

        for j in range(col, size + 1):
            augmented[col][j] /= pivot_value

        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[row][size] for row in range(size)]


def fit_polynomial(data_rows, degree):
    powers = [
        sum((row["bp_rp"] ** power) for row in data_rows)
        for power in range(2 * degree + 1)
    ]
    matrix = []
    vector = []

    for row_index in range(degree + 1):
        matrix.append([powers[row_index + col_index] for col_index in range(degree + 1)])
        vector.append(
            sum(row["teff_k"] * (row["bp_rp"] ** row_index) for row in data_rows)
        )

    return solve_linear_system(matrix, vector)


def predict_temperature(coefficients, bp_rp):
    return sum(coefficient * (bp_rp ** power) for power, coefficient in enumerate(coefficients))


def rmse(data_rows, coefficients):
    squared_errors = []
    for row in data_rows:
        prediction = predict_temperature(coefficients, row["bp_rp"])
        squared_errors.append((prediction - row["teff_k"]) ** 2)
    return math.sqrt(sum(squared_errors) / len(squared_errors))


candidate_degrees = [1, 2, 3, 5]
print("candidate degrees:", candidate_degrees)


In [ ]:
single_split_results = []
for degree in candidate_degrees:
    coefficients = fit_polynomial(train_rows, degree)
    single_split_results.append({
        "degree": degree,
        "train_rmse": rmse(train_rows, coefficients),
        "validation_rmse": rmse(validation_rows, coefficients),
    })

print("Single validation split results:")
for result in single_split_results:
    print(
        f"  degree={result["degree"]}: train RMSE={result["train_rmse"]:.2f}, "
        f"validation RMSE={result["validation_rmse"]:.2f}"
    )

best_single_split = min(single_split_results, key=lambda item: item["validation_rmse"])
print("Best degree on one validation split:", best_single_split["degree"])


In [ ]:
folds = [
    {"S001", "S006", "S011", "S014"},
    {"S002", "S007", "S009", "S013"},
    {"S003", "S004", "S008", "S012"},
]

cv_results = []
for degree in candidate_degrees:
    fold_rmses = []
    for held_out_ids in folds:
        fold_train = [row for row in train_validation_rows if row["source_id"] not in held_out_ids]
        fold_valid = [row for row in train_validation_rows if row["source_id"] in held_out_ids]
        coefficients = fit_polynomial(fold_train, degree)
        fold_rmses.append(rmse(fold_valid, coefficients))
    cv_results.append({
        "degree": degree,
        "fold_rmses": fold_rmses,
        "mean_cv_rmse": sum(fold_rmses) / len(fold_rmses),
    })

print("3-fold cross-validation results:")
for result in cv_results:
    rounded = [round(value, 2) for value in result["fold_rmses"]]
    print(
        f"  degree={result["degree"]}: fold RMSEs={rounded}, "
        f"mean CV RMSE={result["mean_cv_rmse"]:.2f}"
    )

best_cv = min(cv_results, key=lambda item: (item["mean_cv_rmse"], item["degree"]))
print("Best degree from cross-validation:", best_cv["degree"])


In [ ]:
final_degree = best_cv["degree"]
final_coefficients = fit_polynomial(train_validation_rows, final_degree)
final_test_rmse = rmse(test_rows, final_coefficients)

print(f"Final degree chosen after CV: {final_degree}")
print(f"Final test RMSE: {final_test_rmse:.2f} K")

print("Test-set predictions:")
for row in test_rows:
    prediction = predict_temperature(final_coefficients, row["bp_rp"])
    print(
        f"  {row["source_id"]}: true={row["teff_k"]:.0f} K, "
        f"predicted={prediction:.1f} K"
    )


In [ ]:
single_split_degree = best_single_split["degree"]
single_split_coefficients = fit_polynomial(train_validation_rows, single_split_degree)
single_split_test_rmse = rmse(test_rows, single_split_coefficients)

print("Comparison of two selection strategies:")
print(f"  degree chosen by one validation split: {single_split_degree}")
print(f"  degree chosen by cross-validation: {final_degree}")
print(f"  test RMSE after one-split choice: {single_split_test_rmse:.2f} K")
print(f"  test RMSE after cross-validation choice: {final_test_rmse:.2f} K")
